In [ ]:
import json
import re
import sys
import os
sys.path.append("your/path/to/GIMO")  # 替换为 GIMO 项目的实际路径
print("当前工作目录：", os.getcwd())
import argparse
from faiss_rank_utils import get_gt_rank
from sentence_transformers import SentenceTransformer
import faiss


def load_user_items(item_file_path):
    with open(item_file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def extract_gt_title_from_data(dialogue_id_str, user_items):
    try:
        parts = dialogue_id_str.split(",")
        user_id = int(parts[0].split(":")[1].strip())
        item_id = int(parts[1].split(":")[1].strip())
        gt = user_items[user_id]["Items"]
        gt_title = user_items[user_id]["Items"][item_id]["ItemName"]
        return gt_title, gt
    except Exception as e:
        raise ValueError(f"提取 GT 失败: {dialogue_id_str}，错误: {e}")


def read_jsonl_file(file_path):
    data_list = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                data = json.loads(line)
                data_list.append(data)
            except json.JSONDecodeError as e:
                print(f"解析错误：{e}, 行内容: {line[:50]}")
    return data_list


def get_action_type(output_text):
    if not isinstance(output_text, str):
        return "unknown"
    if output_text.startswith("Search["):
        return "search"
    elif output_text.startswith("Ask["):
        return "ask"
    elif output_text.startswith("Recommend["):
        return "recommend"
    elif output_text.startswith("Response["):
        return "response"
    else:
        return "unknown"


def extract_actions(data, user_items, model, index, metadata):
    all_entries = {"search": [], "recommend": [], "ask": []}
    scratchpad = data.get("scratchpad", {})

    try:
        gt_title, ground_truth = extract_gt_title_from_data(data.get("Dialogue_id", ""), user_items)
    except ValueError as e:
        print(f"❗ 跳过该条数据: {e}")
        return all_entries

    query_log = data.get("query_log", {})

    for turn_str, turn_content in scratchpad.items():
        try:
            turn = int(turn_str)
        except ValueError:
            continue

        for key, value in turn_content.items():
            if key.startswith("action "):
                input_text = value.get("input", "").strip()
                output_text = value.get("output", "").strip()
                action_type = get_action_type(output_text)

                if action_type in all_entries:
                    entry = {
                        "system": "You are a helpful assistant",
                        "Instruction": input_text,
                        "input": "",
                        "output": output_text,
                        "type": action_type,
                        "turn": turn,
                        "Dialogue_id": data.get("Dialogue_id"),
                        "gt_title": gt_title,
                        "ground_truth": ground_truth,
                        "is_correct": data.get("is_correct"),
                        "is_recall": data.get("is_recall"),
                        "rec_reward": turn_content.get("rec_reward"),
                        "action_reward": turn_content.get("action_reward"),
                        "exp_reward": turn_content.get("exp_reward"),
                        "action_reason": turn_content.get("action_reason"),
                        "exp_reason": turn_content.get("exp_reason"),
                    }

                    if str(turn) in query_log and str(turn + 1) in query_log:
                        query_turn = query_log.get(str(turn))
                        query_turn_plus_one = query_log.get(str(turn + 1))

                        rank_current = get_gt_rank(query_turn, gt_title, model, index, metadata)
                        rank_next = get_gt_rank(query_turn_plus_one, gt_title, model, index, metadata)
                        rank_diff = rank_current - rank_next
                        entry["rank_diff"] = rank_diff

                    all_entries[action_type].append(entry)

    return all_entries


def main():
    

    domain = "Book"  # 可选值: Book, Yelp, Game
    book_input_file = ""
    yelp_input_file = ""
    game_input_file = ""
    user_item_file = ""
    index_path =""
    metadata_path = ""
    model_path = ""
    output_path =""

    if domain == "Book":
        input_file = book_input_file
    elif domain == "Yelp":
        input_file = yelp_input_file
    elif domain == "Game":
        input_file = game_input_file

    data_list = read_jsonl_file(input_file)
    emb_model = SentenceTransformer(model_path)
    index = faiss.read_index(index_path)
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)

    user_items = load_user_items(user_item_file)
    total_search, total_recommend, total_ask = [], [], []

    for item in data_list:
        extracted = extract_actions(item, user_items, emb_model, index, metadata)
        total_search.extend(extracted["search"])
        total_recommend.extend(extracted["recommend"])
        total_ask.extend(extracted["ask"])

    os.makedirs(output_path, exist_ok=True)

    with open(os.path.join(output_path, 'search_data.json'), 'w', encoding='utf-8') as f:
        json.dump(total_search, f, ensure_ascii=False, indent=4)
    with open(os.path.join(output_path, 'recommend_data.json'), 'w', encoding='utf-8') as f:
        json.dump(total_recommend, f, ensure_ascii=False, indent=4)
    with open(os.path.join(output_path, 'ask_data.json'), 'w', encoding='utf-8') as f:
        json.dump(total_ask, f, ensure_ascii=False, indent=4)

    print(f"✅ 提取完成：Search {len(total_search)} 条，Recommend {len(total_recommend)} 条，Ask {len(total_ask)} 条。")


if __name__ == "__main__":
    main()
